# 02 · Agents with LangChain and LangGraph

**Session 5 — Agent Architecture and Ecosystem Tools · MBRF Data Analytics**

In notebook 01 we wrote the agent loop by hand. Here we rebuild the same agent with **LangChain** and look at what a framework adds:

1. Tools from plain Python functions.
2. `create_agent`: the loop, already written.
3. Switching model provider in one line.
4. Middleware: limits and **human approval** before sensitive tools.
5. Memory across turns.
6. **LangGraph**: when the steps should be fixed by code (a *workflow*) rather than chosen by the model (an *agent*).

> Many tutorials online use the pre-1.0 LangChain API (`AgentExecutor`, `initialize_agent`, chains). This notebook uses LangChain 1.x.

## 0. Setup

In [ ]:
%pip install -q git+https://github.com/albertotb/curso-agentes-ia
# Without git: %pip install -q https://github.com/albertotb/curso-agentes-ia/archive/refs/heads/main.zip

In [ ]:
from typing import Literal

from langchain.agents import create_agent
from langchain.tools import tool

from cai import utils

PROVIDER = "openai"  # "openai" | "gemini"
MODEL = utils.DEFAULT_MODELS[PROVIDER]  # or any other model name of that provider
model = utils.get_chat_model(PROVIDER, MODEL)  # a LangChain chat model: same interface for every provider
print("Using", PROVIDER, MODEL)

# Synthetic bank database used by the tools.
db = utils.load_bank_db()

## 1. Tools from Python functions

With `@tool`, the function **name** becomes the tool name, the **docstring** the description, and the **type hints** the JSON Schema. A `Literal[...]` becomes an `enum`: the model can only choose among those values.

In [ ]:
@tool(parse_docstring=True)
def get_capital_ratios(reference_date: str) -> dict:
    """Get Demo Bank's regulatory capital and liquidity ratios (CET1, Tier 1, total capital, leverage, LCR) at a quarter end.

    Use it for any question about the bank's own capital or liquidity. Ratios are already computed: do not recompute them.

    Args:
        reference_date: Quarter-end date in YYYY-MM-DD format, e.g. 2026-09-30.
    """
    row = db.execute(
        "SELECT * FROM capital_position WHERE reference_date = ?", (reference_date,)
    ).fetchone()
    if row is None:
        available = [
            r[0] for r in db.execute("SELECT reference_date FROM capital_position ORDER BY 1")
        ]
        return {"error": f"No data for {reference_date}. Available quarter ends: {available}"}
    rwa = row["rwa_credit_eur_m"] + row["rwa_market_eur_m"] + row["rwa_operational_eur_m"]
    tier1 = row["cet1_capital_eur_m"] + row["at1_capital_eur_m"]
    return {
        "reference_date": reference_date,
        "total_rwa_eur_m": round(rwa, 1),
        "cet1_ratio_pct": round(100 * row["cet1_capital_eur_m"] / rwa, 2),
        "tier1_ratio_pct": round(100 * tier1 / rwa, 2),
        "total_capital_ratio_pct": round(100 * (tier1 + row["tier2_capital_eur_m"]) / rwa, 2),
        "leverage_ratio_pct": round(100 * tier1 / row["leverage_exposure_eur_m"], 2),
        "lcr_pct": round(100 * row["hqla_eur_m"] / row["net_outflows_30d_eur_m"], 1),
    }


@tool(parse_docstring=True)
def get_customer_loans(customer_id: str) -> dict:
    """Get a customer's profile (segment, internal rating 1=best..8=default) and their loans with outstanding amount, IFRS 9 stage and days past due.

    Args:
        customer_id: Customer ID, e.g. C-0013.
    """
    customer = db.execute(
        "SELECT customer_id, name, segment, internal_rating FROM customers WHERE customer_id = ?",
        (customer_id,),
    ).fetchone()
    if customer is None:
        return {"error": f"Customer {customer_id} not found. IDs look like C-0001."}
    loans = db.execute(
        "SELECT loan_id, product, outstanding_eur, ifrs9_stage, days_past_due FROM loans WHERE customer_id = ?",
        (customer_id,),
    ).fetchall()
    return {**dict(customer), "loans": [dict(r) for r in loans]}


@tool(parse_docstring=True)
def get_portfolio_summary(
    segment: Literal["retail", "sme", "corporate", "all"] = "all",
) -> list[dict]:
    """Outstanding loan amount and number of loans by IFRS 9 stage for a customer segment.

    Args:
        segment: Customer segment to summarise, or "all".
    """
    query = "SELECT l.ifrs9_stage, COUNT(*) AS n_loans, ROUND(SUM(l.outstanding_eur)) AS outstanding_eur FROM loans l JOIN customers c USING (customer_id)"
    params: tuple = ()
    if segment != "all":
        query += " WHERE c.segment = ?"
        params = (segment,)
    return [dict(r) for r in db.execute(query + " GROUP BY l.ifrs9_stage ORDER BY 1", params)]


print(get_portfolio_summary.name)
print(get_portfolio_summary.description)
print(get_portfolio_summary.args)  # the JSON Schema the model will see

## 2. `create_agent`: the loop, already written

Model + tools + system prompt. Underneath it runs exactly the loop from notebook 01.

In [ ]:
SYSTEM_PROMPT = "You are an assistant for Demo Bank's risk department. Answer only from tool results; if the tools cannot answer, say so. Be concise and quote figures with their reference date."
TOOLS = [get_capital_ratios, get_customer_loans, get_portfolio_summary]

agent = create_agent(model, tools=TOOLS, system_prompt=SYSTEM_PROMPT)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "How much of our SME portfolio is in stage 2 or 3, and how did CET1 change between June and September 2026?",
            }
        ]
    }
)
for m in result["messages"]:
    m.pretty_print()

The whole trace is there: the user question, each tool call with its arguments, each tool result, and the final answer. That trace is what you would **log for audit**.

## 3. Switching provider

Because LangChain wraps every provider behind the same interface, the agent does not change. Only the model line does, for example:

```python
model = init_chat_model("openai:gpt-5.4-mini")
model = init_chat_model("anthropic:claude-haiku-5-5")        # pip install langchain-anthropic
model = init_chat_model("google_genai:gemini-3.5-flash")     # pip install langchain-google-genai
```

Useful for comparing quality and cost, or for falling back to another provider when one fails (see `ModelFallbackMiddleware`).

## 4. Middleware: limits and human approval

Middleware hooks into the loop: before or after each model call, around each tool call. LangChain ships several ready-made ones; two matter immediately in a bank:

- **Limits** on model or tool calls, so a confused agent cannot loop (and spend) forever.
- **Human in the loop**: pause before a sensitive tool runs, show the call to a person, and continue only if they approve (or edit / reject it).

Pausing requires saving the agent's state somewhere (a **checkpointer**) so it can resume later — possibly much later, from another process.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware, ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

REVIEW_QUEUE = []


@tool(parse_docstring=True)
def get_cash_transactions(customer_id: str, min_amount_eur: float = 0) -> list[dict]:
    """List a customer's cash transactions in 2026, optionally above a minimum absolute amount.

    Args:
        customer_id: Customer ID, e.g. C-0046.
        min_amount_eur: Minimum absolute amount in EUR.
    """
    rows = db.execute(
        "SELECT tx_id, tx_date, amount_eur, counterparty_country, channel FROM transactions WHERE customer_id = ? AND tx_type = 'cash' AND ABS(amount_eur) >= ? ORDER BY tx_date",
        (customer_id, min_amount_eur),
    )
    return [dict(r) for r in rows]


@tool(parse_docstring=True)
def flag_customer_for_review(customer_id: str, reason: str) -> dict:
    """Open an AML review case for a customer. Use only after checking the customer's transactions.

    Args:
        customer_id: Customer ID, e.g. C-0046.
        reason: Short factual justification citing the transactions.
    """
    REVIEW_QUEUE.append({"customer_id": customer_id, "reason": reason})
    return {"status": "case opened", "customer_id": customer_id}


aml_agent = create_agent(
    model,
    tools=[get_cash_transactions, flag_customer_for_review],
    system_prompt="You are an AML analyst assistant at Demo Bank. Structuring = repeated cash operations just below 10,000 EUR.",
    middleware=[
        ModelCallLimitMiddleware(run_limit=10),
        HumanInTheLoopMiddleware(
            interrupt_on={"flag_customer_for_review": True}
        ),  # reads run freely, the write waits for a human
    ],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "aml-case-1"}}
result = aml_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Check C-0046 for structuring and open a review case if needed.",
            }
        ]
    },
    config,
    version="v2",
)

# The run stopped before flag_customer_for_review. What does it want to do?
for interrupt in result.interrupts:
    for request in interrupt.value["action_requests"]:
        print(request["name"], request["args"])

Now the human decides. Try `approve`, then rerun the two cells and try `reject` with a message.

In [ ]:
decision = {"type": "approve"}
# decision = {"type": "reject", "message": "Not enough evidence: check the counterparty countries first."}

result = aml_agent.invoke(Command(resume={"decisions": [decision]}), config, version="v2")
print(result.value["messages"][-1].text)
print(REVIEW_QUEUE)

## 5. Memory across turns

With a checkpointer, every call with the same `thread_id` continues the same conversation.

In [ ]:
chat_agent = create_agent(
    model, tools=TOOLS, system_prompt=SYSTEM_PROMPT, checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "analyst-42"}}

for question in [
    "What was our LCR at the end of September 2026?",
    "And three months earlier?",
    "Is that trend consistent with the CET1 trend?",
]:
    result = chat_agent.invoke({"messages": [{"role": "user", "content": question}]}, config)
    print(f"> {question}\n{result['messages'][-1].text}\n")

## 6. LangGraph: workflows vs agents

`create_agent` lets **the model** decide the steps. Many banking processes should not work like that: the steps are known and must always happen, in order, and be auditable. That is a **workflow**: the steps are fixed by code, and the LLM is used only where language is needed.

LangGraph builds these as a graph: a shared **state**, **nodes** (plain functions) and **edges** (possibly conditional). `create_agent` itself is a small LangGraph graph.

Example: a quarterly capital check. Fetching and comparing numbers is deterministic; only the memo is written by the LLM, and only when needed.

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

# Illustrative CET1 requirement: Pillar 1 + Pillar 2 requirement (CET1 part) + capital conservation buffer + countercyclical buffer.
CET1_REQUIREMENT_PCT = 4.5 + 1.13 + 2.5 + 0.5
MANAGEMENT_BUFFER_BP = 300


class CapitalCheck(TypedDict, total=False):
    reference_date: str
    ratios: dict
    headroom_bp: float
    status: str
    memo: str


def fetch_ratios(state: CapitalCheck) -> CapitalCheck:
    return {"ratios": get_capital_ratios.invoke({"reference_date": state["reference_date"]})}


def assess(state: CapitalCheck) -> CapitalCheck:
    headroom = round(100 * (state["ratios"]["cet1_ratio_pct"] - CET1_REQUIREMENT_PCT))
    status = "red" if headroom < 0 else "amber" if headroom < MANAGEMENT_BUFFER_BP else "green"
    return {"headroom_bp": headroom, "status": status}


def draft_escalation_memo(state: CapitalCheck) -> CapitalCheck:
    prompt = (
        f"Write a 5-line escalation memo to the CRO. CET1 headroom over the requirement of {CET1_REQUIREMENT_PCT:.2f}% "
        f"is {state['headroom_bp']} bp (status {state['status']}), below the {MANAGEMENT_BUFFER_BP} bp management buffer. "
        f"Figures: {state['ratios']}. Use only these figures."
    )
    return {"memo": model.invoke(prompt).text}


def routine_note(state: CapitalCheck) -> CapitalCheck:
    return {
        "memo": f"{state['reference_date']}: CET1 {state['ratios']['cet1_ratio_pct']}%, headroom {state['headroom_bp']} bp. No action."
    }


graph = StateGraph(CapitalCheck)
graph.add_node("fetch_ratios", fetch_ratios)
graph.add_node("assess", assess)
graph.add_node("draft_escalation_memo", draft_escalation_memo)
graph.add_node("routine_note", routine_note)
graph.add_edge(START, "fetch_ratios")
graph.add_edge("fetch_ratios", "assess")
graph.add_conditional_edges(
    "assess",
    lambda s: "routine_note" if s["status"] == "green" else "draft_escalation_memo",
    ["routine_note", "draft_escalation_memo"],
)
graph.add_edge("draft_escalation_memo", END)
graph.add_edge("routine_note", END)
capital_check = graph.compile()

for date in ["2026-06-30", "2026-09-30"]:
    out = capital_check.invoke({"reference_date": date})
    print(f"--- {date}: {out['status']} ---\n{out['memo']}\n")

In [ ]:
# The graph, drawn (Mermaid). Paste it into https://mermaid.live to render it.
print(capital_check.get_graph().draw_mermaid())

## 7. Exercise: build a credit-risk assistant, step by step

You will build an assistant for Demo Bank's credit-risk team. Each step adds one piece; run your agent after each step to see what changed. Solutions are at the end of the notebook — try first.

Reminder of the tables: `customers(customer_id, name, segment, country, sector, internal_rating, kyc_risk, onboarding_date)`, `loans(loan_id, customer_id, product, principal_eur, outstanding_eur, interest_rate, start_date, maturity_date, ifrs9_stage, days_past_due, collateral)`, `transactions(tx_id, customer_id, tx_date, amount_eur, tx_type, counterparty_country, channel)`. Ratings go from 1 (best) to 8 (default) and are stored as text.

### 7.1 A search tool with constrained parameters

Write `find_customers(segment, min_internal_rating, kyc_risk)`: customers of a segment whose rating is `min_internal_rating` **or worse**, optionally filtered by KYC risk. Use `Literal[...]` for `segment` and `kyc_risk`, and return at most 20 rows (`customer_id`, `name`, `segment`, `internal_rating`, `kyc_risk`).

In [ ]:
@tool(parse_docstring=True)
def find_customers(
    segment: Literal["retail", "sme", "corporate", "all"] = "all",
    min_internal_rating: int = 1,
    kyc_risk: Literal["low", "medium", "high", "any"] = "any",
) -> list[dict]:
    """TODO: describe what the tool returns and when to use it.

    Args:
        segment: TODO
        min_internal_rating: TODO
        kyc_risk: TODO
    """
    # TODO
    ...


# find_customers.invoke({"segment": "sme", "min_internal_rating": 6})

### 7.2 An aggregated tool

Write `get_transaction_summary(customer_id)`. Do **not** return the raw transactions: return totals in and out, the number of cash operations, how many of them are just below 10,000 EUR, and the list of counterparty countries. The model gets what it needs, in a few tokens.

In [ ]:
@tool(parse_docstring=True)
def get_transaction_summary(customer_id: str) -> dict:
    """TODO

    Args:
        customer_id: TODO
    """
    # TODO
    ...


# get_transaction_summary.invoke({"customer_id": "C-0046"})

### 7.3 The agent and its scope

Create `credit_agent` with `get_customer_loans` (section 1) and your two tools. Write a system prompt that:

- says who the assistant serves and what it can answer,
- asks it to cite customer IDs and figures from the tools,
- makes it **refuse** anything outside credit and AML risk.

Test it with the questions below. Does it pick the right tools? Does it refuse the last one?

In [ ]:
CREDIT_SYSTEM_PROMPT = """TODO"""

credit_agent = None  # TODO: create_agent(...)

QUESTIONS = [
    "Which SME customers have an internal rating of 6 or worse? Summarise their loan exposure.",
    "Does C-0042 show anything unusual in its cash activity?",
    "Write a short poem about Madrid.",
]
# for q in QUESTIONS:
#     result = credit_agent.invoke({"messages": [{"role": "user", "content": q}]})
#     print(f"> {q}\n{result['messages'][-1].text}\n")

### 7.4 An action, with human approval

Add a tool `request_credit_limit_review(customer_id, proposed_limit_eur, justification)` that appends the request to a `LIMIT_REVIEWS` list. It changes things, so it must not run without approval: rebuild the agent with `HumanInTheLoopMiddleware` for that tool (and a checkpointer).

Then ask: *"C-0013 has all its loans in stage 2. Request a review to cut its credit limit to 50,000 EUR."* Approve it once; run it again and reject it with a message. What does the agent answer in each case?

In [ ]:
LIMIT_REVIEWS = []

# TODO: tool, agent with middleware + checkpointer, invoke, inspect result.interrupts, resume with Command(...)

### 7.5 Bonus: a second check in the workflow

Extend the quarterly capital check of section 6 so that it also assesses the **LCR**: the requirement is 100 % and the management buffer 10 percentage points. The overall status is the worst of the two (red > amber > green), and the memo must mention which ratio triggered it. Run it for the four quarter ends: which ratios trigger each non-green status?

In [ ]:
# TODO

## When to use what

| | Write the loop yourself | `create_agent` | LangGraph workflow |
|---|---|---|---|
| Who decides the steps | the model | the model | **your code** |
| Good for | learning, small prototypes, full control | assistants with a handful of tools | regulated processes with fixed steps, approvals, audit |
| You get for free | nothing | loop, retries, limits, human approval, memory, provider switch | state, persistence, branching, pause/resume |

A framework does not make the agent smarter: it handles the plumbing. Quality still comes from **tools, data and prompts**.

Other frameworks you will hear about: OpenAI Agents SDK, Google ADK, Claude Agent SDK, PydanticAI, CrewAI, LlamaIndex. Same ideas, different trade-offs.

Next (Friday): give the agent a tool that **searches regulation** — RAG.

---
### Solutions to exercise 7

In [ ]:
# 7.1
@tool(parse_docstring=True)
def find_customers(
    segment: Literal["retail", "sme", "corporate", "all"] = "all",
    min_internal_rating: int = 1,
    kyc_risk: Literal["low", "medium", "high", "any"] = "any",
) -> list[dict]:
    """Find customers by segment, internal rating and KYC risk. Returns at most 20 customers.

    Use it to build lists of customers (e.g. weak ratings, high KYC risk) before looking at each one in detail.

    Args:
        segment: Customer segment, or "all".
        min_internal_rating: Only customers with this rating or worse (1 = best, 8 = default).
        kyc_risk: KYC risk level, or "any".
    """
    query = "SELECT customer_id, name, segment, internal_rating, kyc_risk FROM customers WHERE CAST(internal_rating AS INTEGER) >= ?"
    params: list = [min_internal_rating]
    if segment != "all":
        query += " AND segment = ?"
        params.append(segment)
    if kyc_risk != "any":
        query += " AND kyc_risk = ?"
        params.append(kyc_risk)
    return [dict(r) for r in db.execute(query + " ORDER BY customer_id LIMIT 20", params)]


# 7.2
@tool(parse_docstring=True)
def get_transaction_summary(customer_id: str) -> dict:
    """Summary of a customer's 2026 transactions: totals in and out, cash operations (and how many are just below 10,000 EUR) and counterparty countries.

    Use it to screen a customer for unusual activity before looking at individual transactions.

    Args:
        customer_id: Customer ID, e.g. C-0046.
    """
    row = db.execute(
        """SELECT COUNT(*) AS n_transactions,
                  ROUND(SUM(CASE WHEN amount_eur > 0 THEN amount_eur ELSE 0 END), 2) AS total_in_eur,
                  ROUND(-SUM(CASE WHEN amount_eur < 0 THEN amount_eur ELSE 0 END), 2) AS total_out_eur,
                  SUM(tx_type = 'cash') AS n_cash,
                  SUM(tx_type = 'cash' AND ABS(amount_eur) BETWEEN 9000 AND 9999.99) AS n_cash_just_below_10k
           FROM transactions WHERE customer_id = ?""",
        (customer_id,),
    ).fetchone()
    if row["n_transactions"] == 0:
        return {"error": f"No transactions for {customer_id}. IDs look like C-0001."}
    countries = [
        r[0]
        for r in db.execute(
            "SELECT counterparty_country FROM transactions WHERE customer_id = ? GROUP BY 1 ORDER BY COUNT(*) DESC",
            (customer_id,),
        )
    ]
    return {"customer_id": customer_id, **dict(row), "counterparty_countries": countries}


# 7.3
CREDIT_SYSTEM_PROMPT = """You are an assistant for Demo Bank's credit-risk and AML team.
You answer questions about customers' credit exposure, ratings, IFRS 9 stages and transaction activity, using only the tools.
Always cite customer IDs and the figures returned by the tools. If the tools do not contain the answer, say so.
Politely refuse any request outside credit and AML risk."""

credit_agent = create_agent(
    model,
    tools=[get_customer_loans, find_customers, get_transaction_summary],
    system_prompt=CREDIT_SYSTEM_PROMPT,
)

for q in QUESTIONS:
    result = credit_agent.invoke({"messages": [{"role": "user", "content": q}]})
    print(f"> {q}\n{result['messages'][-1].text}\n")

In [ ]:
# 7.4
@tool(parse_docstring=True)
def request_credit_limit_review(
    customer_id: str, proposed_limit_eur: float, justification: str
) -> dict:
    """Open a request for the credit committee to review a customer's credit limit.

    Args:
        customer_id: Customer ID, e.g. C-0013.
        proposed_limit_eur: Proposed new credit limit in EUR.
        justification: Short factual justification citing loans, stages or ratings.
    """
    LIMIT_REVIEWS.append(
        {
            "customer_id": customer_id,
            "proposed_limit_eur": proposed_limit_eur,
            "justification": justification,
        }
    )
    return {"status": "review requested", "customer_id": customer_id}


credit_agent = create_agent(
    model,
    tools=[
        get_customer_loans,
        find_customers,
        get_transaction_summary,
        request_credit_limit_review,
    ],
    system_prompt=CREDIT_SYSTEM_PROMPT,
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"request_credit_limit_review": True})],
    checkpointer=InMemorySaver(),
)

for i, decision in enumerate(
    [
        {"type": "approve"},
        {
            "type": "reject",
            "message": "Stage 2 alone is not enough: wait for the next rating review.",
        },
    ]
):
    config = {"configurable": {"thread_id": f"limit-review-{i}"}}
    result = credit_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "C-0013 has all its loans in stage 2. Request a review to cut its credit limit to 50,000 EUR.",
                }
            ]
        },
        config,
        version="v2",
    )
    for interrupt in result.interrupts:
        print("Pending:", interrupt.value["action_requests"])
    if result.interrupts:
        result = credit_agent.invoke(
            Command(resume={"decisions": [decision]}), config, version="v2"
        )
    print(f"[{decision['type']}] {result.value['messages'][-1].text}\n")
print(LIMIT_REVIEWS)

In [ ]:
# 7.5
LCR_REQUIREMENT_PCT = 100
LCR_MANAGEMENT_BUFFER_PP = 10
SEVERITY = {"green": 0, "amber": 1, "red": 2}


def status_for(headroom: float, buffer: float) -> str:
    return "red" if headroom < 0 else "amber" if headroom < buffer else "green"


class CapitalLiquidityCheck(CapitalCheck, total=False):
    triggers: list[str]


def assess_capital_and_liquidity(state: CapitalLiquidityCheck) -> CapitalLiquidityCheck:
    ratios = state["ratios"]
    cet1_headroom_bp = round(100 * (ratios["cet1_ratio_pct"] - CET1_REQUIREMENT_PCT))
    lcr_headroom_pp = round(ratios["lcr_pct"] - LCR_REQUIREMENT_PCT, 1)
    statuses = {
        f"CET1 headroom {cet1_headroom_bp} bp (buffer {MANAGEMENT_BUFFER_BP} bp)": status_for(
            cet1_headroom_bp, MANAGEMENT_BUFFER_BP
        ),
        f"LCR headroom {lcr_headroom_pp} pp (buffer {LCR_MANAGEMENT_BUFFER_PP} pp)": status_for(
            lcr_headroom_pp, LCR_MANAGEMENT_BUFFER_PP
        ),
    }
    status = max(statuses.values(), key=SEVERITY.get)
    return {
        "headroom_bp": cet1_headroom_bp,
        "status": status,
        "triggers": [k for k, v in statuses.items() if v != "green"],
    }


def draft_memo_with_triggers(state: CapitalLiquidityCheck) -> CapitalLiquidityCheck:
    prompt = (
        f"Write a 5-line escalation memo to the CRO. Status {state['status']}, triggered by: {state['triggers']}. "
        f"Figures: {state['ratios']}. Use only these figures."
    )
    return {"memo": model.invoke(prompt).text}


graph = StateGraph(CapitalLiquidityCheck)
graph.add_node("fetch_ratios", fetch_ratios)
graph.add_node("assess", assess_capital_and_liquidity)
graph.add_node("draft_escalation_memo", draft_memo_with_triggers)
graph.add_node("routine_note", routine_note)
graph.add_edge(START, "fetch_ratios")
graph.add_edge("fetch_ratios", "assess")
graph.add_conditional_edges(
    "assess",
    lambda s: "routine_note" if s["status"] == "green" else "draft_escalation_memo",
    ["routine_note", "draft_escalation_memo"],
)
graph.add_edge("draft_escalation_memo", END)
graph.add_edge("routine_note", END)
capital_liquidity_check = graph.compile()

for date in ["2025-12-31", "2026-03-31", "2026-06-30", "2026-09-30"]:
    out = capital_liquidity_check.invoke({"reference_date": date})
    print(f"--- {date}: {out['status']} {out.get('triggers', [])} ---\n{out['memo']}\n")